### This is a minimal example for sanitizing sentences. 

In [1]:
from preempt.legacy import LegacyNERShim
from preempt.distilbert_recognizer import DistilBertRecognizer
from preempt.sanitizer import *

/Users/royajavadi/projects/preempt-guardrail/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%load_ext autoreload
%autoreload 2

In [6]:
# Load the span recognizer and adapt it to the sanitizer.
model_path = "/Users/royajavadi/projects/preempt-guardrail/runs/407944/onnx"
recognizer = DistilBertRecognizer(model_path=model_path, 
    max_length=512, 
    backend="onnx",
    onnx_file_name="model.int8.onnx")
ner_model = LegacyNERShim(recognizer)

# Load Sanitizer object
sanitizer = Sanitizer(ner_model, key = "EF4359D8D580AA4F7F036D6F04FC6A94", tweak = "D8E7920AFA330A73")

### Example for sanitizing names of people:

In [7]:
sentences = ["Ben Parker and John Doe went to the bank.", "Who was late today? Adam."]
extracted = ner_model.extract(sentences, entity_type='Name')
print("NER extraction:")
print(extracted)
# Expected output: {'Name': [['Ben Parker', 'John Doe'], ['Adam']]}

NER extraction:
{'Name': [['Parker'], ['Adam']]}


In [8]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity='Name', epsilon=1)
print("Sanitized sentences:")
print(sanitized_sentences)
# Expected output: ['Jay Francois and Lamine Franklin went to the bank.', 'Who was late today? Elie Vinod.']

Sanitized sentences:
['Ben Kian Rodriguez and John Doe went to the bank.', 'Who was late today? Gabe Vinod.']


In [9]:
extracted = ner_model.extract(sanitized_sentences, entity_type='Name')
print("NER extraction:")
print(extracted)
# Expected: {'Name': [['Jay Francois', 'Lamine Franklin'], ['Elie Vinod']]}

NER extraction:
{'Name': [['Kian Rodriguez'], ['Gabe Vinod']]}


In [10]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity='Name')
print("Desanitized sentences:")
print(desanitized_sentences)
# Expected output: ['Ben Parker and John Doe went to the bank.', 'Who was late today? Adam.']

Desanitized sentences:
['Ben Parker and John Doe went to the bank.', 'Who was late today? Adam.']


### Example for sanitizing currency values with FPE:

In [11]:
sentences = ["Ben withdrew $10,000 from the bank. Adam got $5,550 in a loan.", "I won $25 in the lottery."]
extracted = ner_model.extract(sentences, entity_type='Money')
distilbert_extracted = recognizer.recognize(sentences[0])
print(f"DistilBERT extraction:{distilbert_extracted}")
print("NER extraction:")
print(extracted)
# Expected output: {'Money': [['$10,000', '$5,550'], ['$25']]}

DistilBERT extraction:[EntitySpan(start=0, end=3, text='Ben', label='NAME', confidence=0.990349292755127, source='distilbert'), EntitySpan(start=14, end=20, text='10,000', label='MONEY', confidence=1.0, source='currency'), EntitySpan(start=36, end=40, text='Adam', label='NAME', confidence=0.9995546936988831, source='distilbert'), EntitySpan(start=46, end=51, text='5,550', label='MONEY', confidence=1.0, source='currency')]
NER extraction:
{'Money': [['10,000', '5,550'], ['25']]}


In [12]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity='Money', epsilon=1)
print("Sanitized sentences:")
print(sanitized_sentences)
# Expected output: ['Ben withdrew $54829534,343 from the bank. Adam got $0400114,733 in a loan.', 'I won $70096546 in the lottery.']

Sanitized sentences:
['Ben withdrew $54829534,343 from the bank. Adam got $0400114,733 in a loan.', 'I won $70096546 in the lottery.']


In [13]:
extracted = ner_model.extract(sanitized_sentences, entity_type='Money')
print("NER extraction:")
print(extracted)
# Expected output: {'Money': [['54829534,343', '0400114,733'], ['70096546']]}

NER extraction:
{'Money': [['54829534,343', '0400114,733'], ['70096546']]}


In [14]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity='Money')
print("Desanitized sentences:")
print(desanitized_sentences)

Desanitized sentences:
['Ben withdrew $10,000 from the bank. Adam got $5,550 in a loan.', 'I won $25 in the lottery.']


### Example for sanitizing currency values with m-LDP:

In [23]:
sentences = ["Ben withdrew $1000 from the bank. Adam got $555 in a loan.", "I won $25 in the lottery."]
extracted = ner_model.extract(sentences, entity_type='Money')
print("NER extraction:")
print(extracted)

NER extraction:
{'Money': [['1000', '555'], ['25']]}


In [16]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity='Money', 
                                           epsilon=1, use_fpe=False, use_mdp=True)
print("Sanitized sentences:")
print(sanitized_sentences)

Sanitized sentences:
['Ben withdrew $9730 from the bank. Adam got $5590 in a loan.', 'I won $31 in the lottery.']


In [17]:
sanitizer.entity_mapping

[{'9730': '10000', '5590': '5550'}, {'31': '25'}]

In [18]:
extracted = ner_model.extract(sanitized_sentences, entity_type='Money')
print("NER extraction:")
print(extracted)

NER extraction:
{'Money': [['9730', '5590'], ['31']]}


In [19]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity='Money', 
                                          use_fpe=False, use_mdp=True)
print("Desanitized sentences:")
print(desanitized_sentences)

Desanitized sentences:
['Ben withdrew $10000 from the bank. Adam got $5550 in a loan.', 'I won $25 in the lottery.']


### Example for sanitizing age values with m-LDP:

In [20]:
sentences = ["Ben turned 15 years old today.", "I am 25 years old."]
extracted = ner_model.extract(sentences, entity_type='Age')
print("NER extraction:")
print(extracted)

NER extraction:
{'Age': [['15'], ['25']]}


In [21]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity='Age', epsilon=1, use_mdp=True, use_fpe=False)
print("Sanitized sentences:")
print(sanitized_sentences)

Sanitized sentences:
['Ben turned 17 years old today.', 'I am 26 years old.']


In [22]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity='Age')
print("Desanitized sentences:")
print(desanitized_sentences)

Desanitized sentences:
['Ben turned 15 years old today.', 'I am 25 years old.']


In [47]:
sentences = ["Passage: Ben Parker, a 72 Male immigrant, needs your legal views. He earns $34,000 annually. Hamid Aziz is a native here."]
extracted = ner_model.extract(sentences, entity_type='Name')
print("NER extraction:")
print(extracted)
extracted = ner_model.extract(sentences, entity_type='Age')
print("NER extraction:")
print(extracted)
extracted = ner_model.extract(sentences, entity_type='Money')
print("NER extraction:")
print(extracted)


NER extraction:
{'Name': [['Ben Parker', 'Hamid Aziz']]}
NER extraction:
{'Age': [['72']]}
NER extraction:
{'Money': [['34,000']]}


In [48]:
sanitized_sentences, _ = sanitizer.encrypt(sentences, entity='Name', epsilon=1, use_mdp=True, use_fpe=False)
print("Sanitized sentences:")
print(sanitized_sentences)

Sanitized sentences:
['Passage: Warren Marion, a 72 Male immigrant, needs your legal views. He earns $34,000 annually. Joshua Robinson is a native here.']


In [51]:
extracted = ner_model.extract(sanitized_sentences, entity_type='Name')
print("NER extraction:")
print(extracted)

NER extraction:
{'Name': [['Warren Marion', 'Joshua Robinson']]}


In [49]:
desanitized_sentences = sanitizer.decrypt(sanitized_sentences, entity='Name')
print("Desanitized sentences:")
print(desanitized_sentences)

Exception: Decrypt target not found in names dataset!